# The Continuum population model: a guide with live figures

Continuum fills a blank Cardano devnet with simulated users who are born, act, go quiet and leave, at random but realistic rates. This notebook mirrors the published guide section by section. Drag the sliders; the pictures redraw.

Where the guide used illustrations, this notebook uses the **real preprod and preview fits** shipped in `profiles/`, and (when `/data/chain` and `/data/export` are mounted, see `make lab`) the real preprod chain and the seed-7 run. Without them it falls back to tiny test fixtures and says so below. Looking for the calibration workflow itself? See [explore.ipynb](explore.ipynb).

Free companions: [Seeing Theory](https://seeing-theory.brown.edu/) (a visual course in probability), [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html), and [scikit-learn mixtures](https://scikit-learn.org/stable/modules/mixture.html).

**Sections:** 1 Two layers of randomness · 2 Lifecycles and hazard rates · 3 The four distributions · 4 When things happen · 5 Seeds and replay · 6 Calibration · 7 Clustering · 8 Profiles per network · 9 Glossary

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())
sys.path.insert(0, '/work/lab')
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import guide_lib as gl
import guide_figures as gf

PATHS = gl.data_paths()
NETS = gl.networks()
print('data:', PATHS.describe())
print('networks:', {net.name: ('fitted' if net.fitted else 'not fitted yet') for net in NETS})

## 1. Two layers of randomness

Continuum is an *agent-based, discrete-event* simulation: a list of individual actors and a queue of "things that happen next", each with a timestamp. Randomness enters at two speeds, kept apart so "more users" and "busier users" can be turned independently.

- **Population layer** (days to weeks): births, dormancy, retirement. Driven by *hazard rates*.
- **Activity layer** (seconds to hours): wake-ups, a Poisson process with a daily cycle and bursts. An awake actor's agenda then picks an intent (pay, wait, ...).

Nothing changes an actor until the chain confirms it. Below, the bands are slow awake/dormant spells and the ticks are fast wake-ups, for six actors over ten days.

Learn more: [Agent-based model](https://en.wikipedia.org/wiki/Agent-based_model) · [Discrete-event simulation](https://en.wikipedia.org/wiki/Discrete-event_simulation) · [Seeing Theory](https://seeing-theory.brown.edu/)

In [ ]:
interact(gf.show_two_layers, seed=IntSlider(3, 0, 20, description='seed'));

## 2. Lifecycles and hazard rates

A wallet is *Born*, *Active*, *Dormant* (wakes after a log-normal sleep) or *Abandoned*. "When does a wallet go quiet?" is answered by a **hazard rate**: the chance per unit time that it happens now, given it has not happened yet. A constant hazard of one per day gives awake spells of one day *on average*, but any single spell may last an hour or a week: spell lengths are exponential.

Pools have their own cycle (Registered, Active, Retiring, Retired), gated on the chain: the simulator never assumes the chain did something. In the profile files the fields are `hazards.dormancy` and `hazards.dormancy_onset`.

Learn more: [Survival analysis](https://en.wikipedia.org/wiki/Survival_analysis) · [Failure rate (hazard)](https://en.wikipedia.org/wiki/Failure_rate) · [Finite-state machine](https://en.wikipedia.org/wiki/Finite-state_machine) · [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)

In [ ]:
interact(gf.show_lifecycle,
         onset_per_day=FloatSlider(0.5, min=0.1, max=3, step=0.1, description='hazard/day'),
         sleep_median_days=FloatSlider(2, min=0.2, max=10, step=0.2, description='sleep (d)'),
         seed=IntSlider(1, 0, 20));

## 3. The four distributions

A profile is written as distributions, for example `competence: beta(8, 2)`. Each actor draws its own value once, so 1,000 wallets get 1,000 different but realistically spread personalities.

- **Beta**: a percentage-like trait between 0 and 1 (`competence`, `care`). Think of `a` as past successes and `b` as past failures.
- **Log-normal**: durations and amounts. Its logarithm is a bell curve, so it is positive and skewed: `hazards.dormancy`, the body of payment amounts.
- **Pareto**: heavy tails, "a few do most of it" (`rates.activity_tail`, `traits.patience`, payment tails). The smaller alpha, the heavier the tail.
- **Exponential**: the wait until the next event at a constant rate, with no memory. What a hazard produces, and what sits between Poisson ticks.

Learn more: [Beta](https://en.wikipedia.org/wiki/Beta_distribution) · [Log-normal](https://en.wikipedia.org/wiki/Log-normal_distribution) · [Pareto](https://en.wikipedia.org/wiki/Pareto_distribution) · [Exponential](https://en.wikipedia.org/wiki/Exponential_distribution) · [Heavy-tailed](https://en.wikipedia.org/wiki/Heavy-tailed_distribution) · [Seeing Theory: probability distributions](https://seeing-theory.brown.edu/probability-distributions/index.html) · [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)

In [ ]:
ON_RELEASE = dict(continuous_update=False)   # redraw when the slider is let go, not while it moves
interact(gf.show_four_distributions,
         a=FloatSlider(8, min=0.5, max=20, step=0.5, **ON_RELEASE), b=FloatSlider(2, min=0.5, max=20, step=0.5, **ON_RELEASE),
         mu=FloatSlider(1, min=-1, max=2.5, step=0.1, **ON_RELEASE), sigma=FloatSlider(0.5, min=0.1, max=1.5, step=0.05, **ON_RELEASE),
         alpha=FloatSlider(1.3, min=0.5, max=4, step=0.1, **ON_RELEASE), xm=FloatSlider(1, min=0.5, max=3, step=0.25, **ON_RELEASE),
         rate=FloatSlider(1, min=0.2, max=4, step=0.2, **ON_RELEASE));

## 4. When things happen: Poisson, cycles and bursts

An awake actor wakes at random times. The baseline is a **Poisson process**: a steady average rate with exponential gaps. Real usage is not steady, so two effects stack on top:

- **Daily cycle** multiplies the rate up and down (`1 + amplitude * sin(2 pi t / 24h)`).
- **Hawkes bursts**: every event adds `alpha` to the rate, and the boost decays at speed `beta`, so activity clumps like an NFT drop. Continuum requires `alpha < beta`; the ratio `alpha/beta` is the average number of follow-ups an event triggers, and at 1 or more bursts never die out.

A wake-up is not yet a transaction: the actor's agenda usually decides to stay idle. The figure shows the same seed with the layers you switch on.

Learn more: [Poisson point process](https://en.wikipedia.org/wiki/Poisson_point_process) · [Hawkes process](https://en.wikipedia.org/wiki/Hawkes_process) · [Thinning](https://en.wikipedia.org/wiki/Thinning_(statistics)) · [Seeing Theory](https://seeing-theory.brown.edu/) · [scipy.stats.poisson](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.poisson.html)

In [ ]:
interact(gf.show_arrivals,
         rate=FloatSlider(1, min=0.2, max=4, step=0.2, **ON_RELEASE), daily=FloatSlider(0, min=0, max=0.9, step=0.1, **ON_RELEASE),
         alpha=FloatSlider(0, min=0, max=3, step=0.1, **ON_RELEASE), beta=FloatSlider(2, min=0.5, max=5, step=0.1, **ON_RELEASE),
         seed=IntSlider(4, 0, 30));

## 5. Seeds and replay: random, yet repeatable

A computer's random numbers are a deterministic sequence fixed by a starting **seed**. Continuum takes one root seed per run and splits it into an independent stream per path (`actor/7/activity`, a pool's KES key, ...) with a key-derivation function, **HKDF**. Two consequences the whole project relies on:

1. **Same seed, same run.** A bug found on Tuesday replays byte for byte on Friday.
2. **Adding an actor changes nobody else.** Actor 31 has its own stream, so actors 1 to 30 behave as before. That keeps bisecting across node builds meaningful.

The cell below derives streams the same way (HKDF-SHA256 over the path; the engine feeds ChaCha8, this toy uses PCG64) and checks both properties.

Learn more: [Pseudorandom number generator](https://en.wikipedia.org/wiki/Pseudorandom_number_generator) · [HKDF](https://en.wikipedia.org/wiki/HKDF) · [Random seed](https://en.wikipedia.org/wiki/Random_seed) · [numpy Generator](https://numpy.org/doc/stable/reference/random/generator.html)

In [ ]:
interact(gl.print_replay, seed=IntSlider(7, 0, 99, description='root seed'), actors=IntSlider(5, 2, 12));

## 6. Calibration: fit, run, grade

Made-up numbers make a made-up chain. So the lab measures real behaviour from a converted chain (`mithril2pq` Parquet), **fits** profile settings (`fit.py`), **runs** the simulation (`continuum run --fake`, `continuum export`), and **grades** the simulated chain with the same statistics (`grade.py`) against the saved reference (`target.json`). Where they disagree, the profile is wrong.

Fitting picks parameters that match data: maximum likelihood for value distributions, method of moments for the beta traits.

**Grading with the KS statistic.** Plot each sample as a cumulative curve (the share at or below x). **D** is the largest vertical gap between the two curves: 0 is identical, 1 is disjoint. A statistic **passes when D <= 0.1**. Why not a p-value? With thousands of samples a p-value calls even a trivial difference significant; D measures how big the difference is, which is what matters. Try it: slide the shift and the sample size.

Learn more: [Empirical distribution function](https://en.wikipedia.org/wiki/Empirical_distribution_function) · [Kolmogorov-Smirnov test](https://en.wikipedia.org/wiki/Kolmogorov%E2%80%93Smirnov_test) · [Maximum likelihood](https://en.wikipedia.org/wiki/Maximum_likelihood_estimation) · [Effect size](https://en.wikipedia.org/wiki/Effect_size) · [scipy.stats.ks_2samp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ks_2samp.html) · [Seeing Theory](https://seeing-theory.brown.edu/)

In [ ]:
interact(gf.show_ks_demo,
         shift=FloatSlider(0.15, min=0, max=1, step=0.05, **ON_RELEASE),
         n=IntSlider(2000, 100, 20000, step=100, **ON_RELEASE));

### Real data: the seed-7 run against the preprod target

`continuum run --fake --seed 7 --wallets 100 --epochs 2 --profiles preprod` simulates 100 wallets over 90 days. Each panel overlays the run's samples on the preprod target. Wherever the curves part by more than 0.1, the profile needs work; the known gaps are the engine's 2 ADA payment floor, gaps that are exponential where the chain's are bursty, and a type mix that is all payments.

In [ ]:
run_samples, overall_target, target_path = gl.run_and_target(PATHS)
print('run:', PATHS.export, '| target:', os.path.basename(target_path), '| real run' if PATHS.real else '| fixture run (not the seed-7 preprod run)')
gf.fig_ks(run_samples, overall_target); plt.show()

## 7. Clustering: many kinds of user, not one average

One profile fitted to everyone describes an "average user" who does not exist. Clustering first sorts units into groups that behave alike, then fits a profile per group.

- **What is one "user"?** A wallet spreads its funds over many addresses that share one *stake credential*, so a **unit** is the stake credential where there is one and the address otherwise.
- **Features** per unit: transactions per day, median gap, median payment, in/out counts, active days, longest dormancy (log-scaled, then standardised). Units that mostly call scripts or send staking certificates are split off first by absolute thresholds, because shares pinned at exactly 0 or 1 make a Gaussian mixture chase tiny-variance components.
- **Gaussian mixture**: assume the cloud of units is overlapping blobs and find them. **BIC** scores a fit by how well it explains the data minus a penalty for each extra blob; the lab searches k = 2..15 and takes the elbow of the curve.
- **Weighting**: by number of units gives a realistic population; by number of transactions gives realistic traffic. A few busy bots make the two differ.

Cluster names come from absolute thresholds in real units (see `lab/README.md`): `holder` below 0.05 tx/day, `occasional-payer` up to 1 tx/day, `frequent-trader` above, `script-user` and `staker` when at least half the unit's transactions are script calls or staking certificates; a size suffix (`-small` under 10 ADA, `-large` from 1,000 ADA median payment) separates repeats.

Learn more: [Cluster analysis](https://en.wikipedia.org/wiki/Cluster_analysis) · [Mixture model](https://en.wikipedia.org/wiki/Mixture_model) · [Bayesian information criterion](https://en.wikipedia.org/wiki/Bayesian_information_criterion) · [scikit-learn: Gaussian mixtures](https://scikit-learn.org/stable/modules/mixture.html) · [Seeing Theory](https://seeing-theory.brown.edu/)

In [ ]:
gl.print_one_wallet_one_unit()

In [ ]:
units_table = gl.units_table(PATHS)                      # one row per unit: unit_features.COLUMNS
clustering = gl.cluster_units_table(units_table, PATHS)  # adds the `cluster` column
print(gl.describe_clustering(units_table, clustering))
print(gl.cluster_summary(units_table))

In [ ]:
gf.fig_units_scatter(units_table, PATHS.real).show()

In [ ]:
gf.fig_weighting(*gl.cluster_weights(units_table)); plt.show()
gf.fig_bic(*gf.bic_curves(clustering.fit, NETS[:2])); plt.show()

## 8. What each network looks like

The fitted profile files in `profiles/` hold one profile per cluster, each with a weight; `continuum run --profiles preprod` splits wallets by those weights. Below, the preview and preprod fits side by side, then the same views for mainnet, which shows **not fitted yet** until `profiles/mainnet.yaml` exists and fills itself in then.

Learn more: [Probability density function](https://en.wikipedia.org/wiki/Probability_density_function) · [Cumulative distribution function](https://en.wikipedia.org/wiki/Cumulative_distribution_function) · [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)

In [ ]:
for net in NETS:
    print(net.summary())
gf.fig_shares(NETS); plt.show()

In [ ]:
gf.fig_quantile_cdfs(NETS, 'activity_per_day', 'transactions per day per unit', 'Activity (per cluster)', logx=True); plt.show()
gf.fig_quantile_cdfs(NETS, 'log10_gap_seconds', 'log10 seconds between a unit\'s transactions', 'Gaps between transactions'); plt.show()

In [ ]:
gf.fig_amounts(NETS); plt.show()
gf.fig_mix(NETS); plt.show()
gf.fig_traits(NETS); plt.show()

## 9. Glossary

| term | meaning |
|---|---|
| distribution | how likely each possible value is |
| PDF / density | the curve whose height shows where values are common |
| CDF | cumulative curve: the share of values at or below x; the KS test compares CDFs |
| hazard rate | chance per unit time that an event happens now, given it has not yet |
| Poisson process | events at a steady average rate with no memory; gaps are exponential |
| Hawkes process | a Poisson process where each event temporarily raises the rate, causing bursts (needs alpha < beta) |
| heavy tail | extreme values are rare but far more common than a bell curve predicts |
| seed | the starting number that fixes a pseudo-random sequence: same seed, same run |
| HKDF | key-derivation function that splits one seed into independent per-path streams |
| fit | choosing a distribution's parameters to match observed data |
| KS statistic D | the largest gap between two cumulative curves; passes at D <= 0.1 |
| unit | the stake credential of an address (the address when it has none): one "user" |
| cluster | a group of units that behave alike; each gets its own profile and weight |
| BIC | model score that rewards fit and penalises extra clusters; lower is better |
| k | the number of mixture components (clusters) |
| k deep | a block with k blocks on top can no longer be rolled back; only that state is trusted |

More: [Seeing Theory](https://seeing-theory.brown.edu/) · [Monte Carlo methods](https://en.wikipedia.org/wiki/Monte_Carlo_method) · [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html) · [scikit-learn mixtures](https://scikit-learn.org/stable/modules/mixture.html)